# 133 相補 미성형 지도 전문가: 군집 SVM + 조건비교 물리피처
A: std24 군집 전문가/전체 SVM 기하평균. B: 조건매칭physical12 RFETLR 평균. 판정선은 innerF1 정상FP3% 제한. fixedOR는 외부 결과로 선택한 개발후보. inner_selected는 매 외부Run을 제외한 안쪽 검증에서 A/B/OR 선택. 전체 탐색은 이미 본 자료를 사용했으므로 신규 데이터 검증과 구분한다.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd();t=root/'reports/tables'
nsA={};nsB={}
exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\nfrom sklearn.cluster import KMeans\nmemo={};audit=[]\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for view in ['std24']:\n  X=views[view];imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[ref][normal])\n  a=imp.transform(X[ref]);b=imp.transform(X[q]);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  full=SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,yy)\n  fullscore=expit(full.decision_function(b))\n  pos=np.flatnonzero(yy==1);neg=np.flatnonzero(normal)\n  k=min(2,len(np.unique(a[pos],axis=0)))\n  modes=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(a[pos]) if k>1 else np.zeros(len(pos),int)\n  experts=[]\n  for mode in np.unique(modes):\n   chosen=np.r_[neg,pos[modes==mode]]\n   scores=[]\n   for C in [.1,1.]:\n    model=SVC(C=C,class_weight='balanced',gamma='scale').fit(a[chosen],yy[chosen])\n    scores.append(expit(model.decision_function(b)))\n   for C in [.1,1.]:\n    model=LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(a[chosen],yy[chosen])\n    scores.append(model.predict_proba(b)[:,1])\n   experts.append(np.mean(scores,axis=0))\n  expert=np.max(experts,axis=0)\n  out[view+'_mode_max']=expert\n  out[view+'_full']=fullscore\n  out[view+'_geometric']=np.sqrt(expert*fullscore)\n  out[view+'_agreement_min']=np.minimum(expert,fullscore)\n  out[view+'_union_max']=np.maximum(expert,fullscore)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n",nsA)
exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\nfrom sklearn.neighbors import NearestNeighbors\nmemo={};audit=[]\ncontext=lab[[v+'_raw' for v in ['Barrel_Temperature_5','Mold_Temperature_3','Hopper_Temperature','Max_Injection_Speed']]].to_numpy(float)\ndef localize(ref,qq,X):\n normal=ref[y[ref]==0]\n imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[normal])\n xx=imp.transform(X)\n ci=SimpleImputer(strategy='median').fit(context[normal])\n cx=ci.transform(context);cs=RobustScaler().fit(cx[normal]);cx=cs.transform(cx)\n nn=NearestNeighbors(n_neighbors=min(31,len(normal))).fit(cx[normal])\n near=nn.kneighbors(cx[qq],return_distance=False)\n floor=np.maximum(xx[normal].std(axis=0)*.2,1e-6)\n out=[]\n for i,ids in zip(qq,near):\n  peers=normal[ids];peers=peers[peers!=i][:30]\n  out.append((xx[i]-np.median(xx[peers],axis=0))/np.maximum(xx[peers].std(axis=0),floor))\n return np.asarray(out)\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for tag,X in [('localphysical12',views['physical12'])]:\n  a=localize(ref,ref,X);b=localize(ref,q,X)\n  sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  for name,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:\n   pp=[]\n   for seed in [42,7]:\n    m=cls(n_estimators=120,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)\n    pp.append(m.predict_proba(b)[:,1])\n   out[tag+'_'+name]=np.mean(pp,axis=0)\n  for C in [.1,1.]:\n   m=SVC(C=C,gamma='scale',class_weight='balanced').fit(a,yy)\n   out[tag+'_svm'+str(C)]=expit(m.decision_function(b))\n  m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)\n  out[tag+'_lr']=m.predict_proba(b)[:,1]\n  out[tag+'_mean']=np.mean([out[tag+'_'+n] for n in ['rf','et','svm0.1','svm1.0','lr']],axis=0)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n",nsB)
lab=nsA['lab'];y=nsA['y'];g=nsA['g'];use=nsA['use']
assert lab.TimeStamp.equals(nsB['lab'].TimeStamp)
records=[];choices=[]
for block in pd.unique(g):
 ref=np.flatnonzero((g!=block)&use);q=np.flatnonzero(g==block)
 sa=nsA['predict'](ref,q)['std24_geometric']
 sb=nsB['predict'](ref,q)['localphysical12_mean']
 ia=np.full(len(ref),np.nan);ib=np.full(len(ref),np.nan)
 for held in pd.unique(g[ref]):
  mask=g[ref]==held
  ia[mask]=nsA['predict'](ref[~mask],ref[mask])['std24_geometric']
  ib[mask]=nsB['predict'](ref[~mask],ref[mask])['localphysical12_mean']
 ca=nsA['threshold'](y[ref],ia,.03);cb=nsB['threshold'](y[ref],ib,.03)
 inner={'A':ia>=ca,'B':ib>=cb,'OR':(ia>=ca)|(ib>=cb)}
 outer={'A':sa>=ca,'B':sb>=cb,'OR':(sa>=ca)|(sb>=cb)}
 def measure(yy,pr):
  tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
  return int(tp),int(fp),int(fn),int(tn),2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
 stats={k:measure(y[ref],v) for k,v in inner.items()}
 selected=max(stats,key=lambda k:(stats[k][4],stats[k][0],-stats[k][1],k!='OR'))
 choices.append(dict(block=block,selected=selected,innerA_F1=stats['A'][4],innerB_F1=stats['B'][4],innerOR_F1=stats['OR'][4],thresholdA=ca,thresholdB=cb))
 for model,pred in [('fixed_OR',outer['OR']),('inner_selected',outer[selected]),('expert_A',outer['A']),('matched_B',outer['B'])]:
  for j,i in enumerate(q):records.append(dict(model=model,TimeStamp=lab.TimeStamp.iloc[i],run_id=block,type=lab.type.iloc[i],actual=int(y[i]),prediction=int(pred[j]),A=int(outer['A'][j]),B=int(outer['B'][j])))
 print('completed',block,selected,flush=True)
p=pd.DataFrame(records);rows=[]
for n,z in p.groupby('model'):
 z=z[z.type.isin(['정상','미성형'])]
 tp,fp,fn,tn,f=measure(z.actual,z.prediction)
 rows.append(dict(model=n,TP=tp,FP=fp,FN=fn,TN=tn,F1=f))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'choices')]:obj.to_csv(t/f'rg3_shortshot_133_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print(pd.DataFrame(choices).to_string(index=False))


completed rg3_R1 A


completed rg3_R2 A


completed rg3_R3 A


completed rg3_R4 OR


completed rg3_R5 A


         model  TP  FP  FN  TN       F1
      expert_A   2   8   6 558 0.222222
      fixed_OR   3   9   5 557 0.300000
inner_selected   2   8   6 558 0.222222
     matched_B   1   2   7 564 0.181818
 block selected  innerA_F1  innerB_F1  innerOR_F1  thresholdA  thresholdB
rg3_R1        A   0.000000   0.000000    0.000000    0.796410    0.660689
rg3_R2        A   0.153846   0.000000    0.142857    0.724654    0.678391
rg3_R3        A   0.000000   0.000000    0.000000    0.610767    0.496709
rg3_R4       OR   0.210526   0.222222    0.285714    0.592397    0.635051
rg3_R5        A   0.222222   0.000000    0.200000    0.718318    0.685121
